In [0]:
CATALOG = "adb_retailedge_dev"
SCHEMA  = "healthcare_cms"

SILVER_PROVIDER = f"{CATALOG}.{SCHEMA}.silver_provider_summary"
SILVER_DRUGS     = f"{CATALOG}.{SCHEMA}.silver_drugs"
SILVER_INPATIENT = f"{CATALOG}.{SCHEMA}.silver_inpatient"

print(f"Source 1 : {SILVER_PROVIDER}")
print(f"Source 2 : {SILVER_DRUGS}")
print(f"Source 3 : {SILVER_INPATIENT}")


Source 1 : adb_retailedge_dev.healthcare_cms.silver_provider_summary
Source 2 : adb_retailedge_dev.healthcare_cms.silver_drugs
Source 3 : adb_retailedge_dev.healthcare_cms.silver_inpatient


In [0]:

# gold_provider_pertoformance

from pyspark.sql.functions import count, round as spark_round
from pyspark.sql.functions import sum as spark_sum, avg as spark_avg
from pyspark.sql.window import Window
from  pyspark.sql.functions import rank

df_providers = spark.table(SILVER_PROVIDER)

# THIS TABLE ASNWERS: WHICH SPECIALISTS AND STATES COST MEDICARE THE MOST ?


# Aggregate by state + provider_type 

gold_provider_performance = df_providers \
    .groupBy("state", "provider_type")\
    .agg(
        count("provider_npi").alias("total_providers"),
        spark_round(spark_avg("avg_medicare_payment"), 2).alias("avg_medicare_payment"),
        spark_round(spark_avg("avg_submitted_charge"), 2).alias("avg_submitted_charge"),
        spark_round(spark_sum("total_services"), 0).alias("total_services"),
        spark_round(spark_sum("total_beneficiaries"), 0).alias("total_beneficiaries")
    )

# Rank by avg_medicare_payment within each state
window_spec = Window.partitionBy("state").orderBy(
    gold_provider_performance["avg_medicare_payment"].desc()
)

gold_provider_performance = gold_provider_performance \
    .withColumn("rank_in_state", rank().over(window_spec))

gold_provider_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_provider_performance")

count = spark.table(f"{CATALOG}.{SCHEMA}.gold_provider_performance").count()
print(f"gold_provider_performance: {count:,} rows written")


gold_provider_performance: 3,435 rows written


In [0]:
# gold_drug_spend_analysis

# This table answers: Which drugs cost Medicare the most? Which prices are rising fastest?

from pyspark.sql.functions import col, round as spark_round, when

df_drugs = spark.table(SILVER_DRUGS)

gold_drug_spend_analysis = df_drugs \
    .select(
        "brand_name",
        "generic_name",
        "manufacturer",
        "total_claims_2024",
        "total_beneficiaries_2024",
        spark_round(col("total_spending_2024"), 2).alias("total_spending_2024"),
        spark_round(col("avg_spend_per_claim_2024"), 2).alias("avg_spend_per_claim_2024"),
        spark_round(col("avg_spend_per_bene_2024"), 2).alias("avg_spend_per_bene_2024"),
        spark_round(col("total_spending_2023"), 2).alias("total_spending_2023"),
        spark_round(col("total_spending_2022"), 2).alias("total_spending_2022"),
        spark_round(col("cagr_2020_2024"), 4).alias("cagr_2020_2024"),
        "outlier_flag",
        when(col("cagr_2020_2024") > 0.10, "High Growth")
        .when(col("cagr_2020_2024") > 0.05, "Moderate Growth")
        .when(col("cagr_2020_2024") <= 0.05, "Stable")
        .otherwise("Unknown").alias("price_trend_category")
    ) \
    .filter(col("total_spending_2024").isNotNull()) \
    .orderBy(col("total_spending_2024").desc())

gold_drug_spend_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_drug_spend_analysis")
    
count = spark.table(f"{CATALOG}.{SCHEMA}.gold_drug_spend_analysis").count()
print(f"gold_drug_spend_analysis: {count:,} rows written")

gold_drug_spend_analysis: 14,536 rows written


In [0]:
# gold_diagnosis_trends

# This table answers: Which diseases are most common state by state? Which states have the highest hospital costs?

from pyspark.sql.functions import col, round as spark_round, rank
from pyspark.sql.window import Window

df_inpatient = spark.table(SILVER_INPATIENT)

gold_diagnosis_trends = df_inpatient \
      .groupBy("geo_code", "geo_description", "drg_code", "drg_description") \
      .agg(
          spark_round(spark_sum("total_discharges"), 0).alias("total_discharges"),
          spark_round(spark_avg("avg_medicare_payment"), 2).alias("avg_medicare_payment"),
          spark_round(spark_avg("avg_submitted_charge"), 2).alias("avg_submitted_charge"),
          spark_round(spark_avg("avg_total_payment"), 2).alias("avg_total_payment")
      )

  # Rank diagnoses within each state by total discharges
window_spec = Window.partitionBy("geo_code").orderBy(
      col("total_discharges").desc()
  )

gold_diagnosis_trends = gold_diagnosis_trends \
      .withColumn("rank_in_state", rank().over(window_spec))

gold_diagnosis_trends.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_diagnosis_trends")

count = spark.table(f"{CATALOG}.{SCHEMA}.gold_diagnosis_trends").count()
print(f"gold_diagnosis_trends: {count:,} rows written")

gold_diagnosis_trends: 25,804 rows written


In [0]:
# gold_fraud_indicators

# This table answers: Which doctors have abnormal billing patterns that could indicate fraud?

# The logic flags providers where their average payment is significantly higher than the average for their specialty and state.

from pyspark.sql.functions import col, round as spark_round, avg as spark_avg, stddev, when

df_providers = spark.table(SILVER_PROVIDER)

  # Calculate average and stddev of medicare payment per state + specialty
specialty_stats = df_providers \
      .groupBy("state", "provider_type") \
      .agg(
          spark_round(spark_avg("avg_medicare_payment"), 2).alias("specialty_avg_payment"),
          spark_round(stddev("avg_medicare_payment"), 2).alias("specialty_stddev_payment")
      )

# Join back to provider level
df_joined = df_providers.join(specialty_stats, on=["state", "provider_type"], how="left")

  # Flag providers who are more than 2 standard deviations above specialty average
gold_fraud_indicators = df_joined \
      .withColumn("deviation_from_avg",
          spark_round(col("avg_medicare_payment") - col("specialty_avg_payment"), 2)
      ) \
      .withColumn("stddev_multiplier",
          spark_round(
              (col("avg_medicare_payment") - col("specialty_avg_payment")) /
              col("specialty_stddev_payment"), 2
          )
      ) \
      .withColumn("fraud_risk_flag",
          when(col("stddev_multiplier") > 2, "High Risk")
          .when(col("stddev_multiplier") > 1, "Medium Risk")
          .otherwise("Normal")
      ) \
      .filter(col("fraud_risk_flag") != "Normal") \
      .select(
          "provider_npi",
          "provider_name",
          "provider_first_name",
          "city",
          "state",
          "provider_type",
          "avg_medicare_payment",
          "specialty_avg_payment",
          "deviation_from_avg",
          "stddev_multiplier",
          "fraud_risk_flag",
          "total_services",
          "total_beneficiaries"
      ) \
      .orderBy(col("stddev_multiplier").desc())

gold_fraud_indicators.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_fraud_indicators")

count = spark.table(f"{CATALOG}.{SCHEMA}.gold_fraud_indicators").count()
print(f"gold_fraud_indicators: {count:,} rows written")

gold_fraud_indicators: 9,184 rows written


In [0]:
# Verify all Gold tables

gold_tables = [
      "gold_provider_performance",
      "gold_drug_spend_analysis",
      "gold_diagnosis_trends",
      "gold_fraud_indicators"
  ]

total = 0
for table in gold_tables:
      count = spark.table(f"{CATALOG}.{SCHEMA}.{table}").count()
      total += count
      print(f"{table:<35}: {count:>10,} rows")

print(f"\nTOTAL Gold rows: {total:,}")

gold_provider_performance          :      3,435 rows
gold_drug_spend_analysis           :     14,536 rows
gold_diagnosis_trends              :     25,804 rows
gold_fraud_indicators              :      9,184 rows

TOTAL Gold rows: 52,959
